In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))

from pathlib import Path
import re
import torch as th
from imitation.data import rollout
from imitation.data.types import Trajectory
from imitation.data import rollout
import numpy as np
import gymnasium as gym
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
import torch.nn as nn
from imitation.util import logger as imit_logger
from torch.utils.data import DataLoader
from imitation.data.types import Transitions
import datetime
from imitation.algorithms.adversarial.gail import GAIL
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from final_fight import FinalFightActionWrapper, make_env
from stable_baselines3 import PPO

from final_fight import TemporalAttentionLSTM
import gc
from IPython import get_ipython
import cv2

from utils import get_last_index

current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"./tensorboard_logs/run_{current_time}"
custom_logger = imit_logger.configure(log_path, ["tensorboard", "stdout"])

gc.collect()
th.cuda.empty_cache()

rng = np.random.default_rng()

demo_path = 'demos/'

train_path = 'gail/'
sub_train_path = train_path + "steps/"

os.makedirs(train_path, exist_ok=True)
os.makedirs(sub_train_path, exist_ok=True)


ENT_WEIGHT= 1e-5 # 1e-3 # 0 # 1e-4 # 0.001 # 0.01 # 0 # 1e-3 # 0 # 1e-4
BATCH_SIZE= 512 # 512 # 128 # 128 # 256 # 64 # 256 # 128 # 64 # 128 # 32 # 64 # 128
LEARNING_RATE= 1e-5 # 1e-4 # 4e-4  # 1e-4 # 5e-5 # 1e-4 # 2e-4
NUMBER_TRAIN = 200

action_space = gym.spaces.MultiBinary(6)

observation_space = gym.spaces.Box(
    low=0,
    high=255,
    shape=(96, 96, 1), # 1 frame 96x96
    dtype=np.uint8,
)


last_index = get_last_index(demo_path, "demos", "pt")

print("last_index: " + str(last_index))

class GAILNoShuffle(GAIL):
    def _make_data_loader(self, transitions, batch_size):
        dataset = self._make_dataset(transitions)
        
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=True,
        )

def clean_invalid_logic(acts):
    # Ensures that in the training dataset has only valid moves
    # prevent turn left and right, or move down and up simultaneously
    
    # Up (0) and Down (1)
    vertical_conflict = (acts[:, 0] > 0.5) & (acts[:, 1] > 0.5)
    acts[vertical_conflict, 0:2] = 0

    # Left (2) and Right (3)
    horizontal_conflict = (acts[:, 2] > 0.5) & (acts[:, 3] > 0.5)
    acts[horizontal_conflict, 2:4] = 0
    return acts

def manual_flatten_trajectories(trajectories):
    all_obs = []
    all_acts = []
    all_next_obs = []
    all_dones = []
    all_infos = []

    for traj in trajectories:
        all_obs.append(traj.obs[:-1])
        
        all_acts.append(traj.acts)
        
        all_next_obs.append(traj.obs[1:])
        
        dones = np.zeros(len(traj.acts), dtype=bool)
        dones[-1] = True
        all_dones.append(dones)
        
        if traj.infos is not None:
            all_infos.extend(traj.infos)
        else:
            all_infos.extend([{}] * len(traj.acts))

    return Transitions(
        obs=np.concatenate(all_obs, axis=0),
        acts=np.concatenate(all_acts, axis=0),
        next_obs=np.concatenate(all_next_obs, axis=0),
        dones=np.concatenate(all_dones, axis=0),
        infos=np.array(all_infos)
    )

def split_into_traj(trajectories, num_parts=4):
    split_trajs = []
    for traj in trajectories:
        total_frames = len(traj.acts)
        part_duration = total_frames // num_parts
        
        for i in range(num_parts):
            start = i * part_duration
            end = (i + 1) * part_duration if i < (num_parts - 1) else total_frames
            
            chunk_obs = traj.obs[start : end + 1]
            chunk_acts = traj.acts[start : end]
            
            split_trajs.append(
                Trajectory(
                    obs=chunk_obs,
                    acts=chunk_acts,
                    infos=traj.infos[start : end] if traj.infos else None,
                    terminal=(i == num_parts - 1)
                )
            )
    return split_trajs

class CustomActorCriticPolicy(ActorCriticPolicy):
    def __init__(self, observation_space, action_space, lr_schedule, *args, **kwargs):

        kwargs.pop("features_extractor_class", None)
        kwargs.pop("features_extractor_kwargs", None)
        kwargs.pop("net_arch", None)
    
        super().__init__(
            observation_space,
            action_space,
            lr_schedule,
            *args,
            **kwargs,
            features_extractor_class=TemporalAttentionLSTM,
            features_extractor_kwargs=dict(features_dim=256),
            net_arch=dict(pi=[256, 256], vf=[256, 256])
        )
        self._last_dones = None
        self.retain_graph = True
    
    def forward(self, obs, deterministic=False):
        if hasattr(self.features_extractor, 'reset_hidden'):
            if self._last_dones is not None:
                self.features_extractor.reset_hidden(self._last_dones)
        
        return super().forward(obs, deterministic)
    
    def predict(self, observation, state=None, episode_start=None, deterministic=False):
        if episode_start is not None:
            self._last_dones = episode_start
        
        return super().predict(observation, state, episode_start, deterministic)


policy = CustomActorCriticPolicy(
    observation_space=observation_space,
    action_space=action_space,
    lr_schedule=lambda _: th.finfo(th.float32).max,  # gail control the learning rate
)


th.serialization.add_safe_globals([Trajectory])

env = make_vec_env(make_env(human=True), n_envs=1, vec_env_cls=SubprocVecEnv)

last_index_imitation = get_last_index(str(sub_train_path), "bc_policy", "zip")

print(f"Last index gail: {last_index_imitation}")

learner = None

if last_index_imitation < 0:
    learner = PPO(
        policy=CustomActorCriticPolicy, 
        env=env,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        ent_coef=ENT_WEIGHT,
        n_steps=2048,
        verbose=1,
        device="cuda"
    )
    
    
    loaded_policy = CustomActorCriticPolicy.load(f"./imitation/steps/bc_policy36.zip", device="cuda")
    learner.policy.load_state_dict(loaded_policy.state_dict())
else:
    learner = PPO.load(f"./gail/steps/gail_policy{last_index_imitation}.zip", device="cuda")

reward_net = BasicRewardNet(
    observation_space=env.observation_space,
    action_space=env.action_space,
    normalize_input_layer=RunningNorm,
)

files_index = np.arange(last_index + 1)

np.random.shuffle(files_index)

def fix_action_format(acts):
    """
    Fix action shape
    """
    if isinstance(acts, np.ndarray):
        if acts.ndim == 3 and acts.shape[1] == 1:
            acts = acts.squeeze(1)
        
        acts = acts.astype(np.float32)
    
    return acts

def fix_trajectories(trajectories, aug_prob=0.5, transpose=True):
    fixed_trajectories = []

    for traj in trajectories:
        obs = np.array(traj.obs)

        acts = fix_action_format(np.array(traj.acts, dtype=np.int8))

        # remove invalid moves
        acts = clean_invalid_logic(acts)


        # Case (T, 1, H, W, C)
        if obs.ndim == 5 and obs.shape[1] == 1:
            obs = obs[:, 0]  # remove dimension 1 → (T, H, W, C)

        # Case HWC → CHW
        if obs.shape[-1] == 1 and transpose: # obs.shape[-1] == 4
            obs = obs.transpose(0, 3, 1, 2)  # (T, 1, H, W)

        if obs.shape[0]>= BATCH_SIZE * 10:
            print("obs shape:", obs.shape)

            fixed_trajectories.append(
                Trajectory(
                    obs=obs,
                    acts=acts,
                    infos=traj.infos,
                    terminal=traj.terminal
                )
            )

    return fixed_trajectories

buffer = []

for idx, file_idx in enumerate(files_index):
    trajectories = th.load(demo_path + f"demos{file_idx}.pt", weights_only=False)
    fixed_trajectories = fix_trajectories(trajectories, transpose=False)
    np.random.shuffle(fixed_trajectories)
    buffer.extend(fixed_trajectories)

transitions = manual_flatten_trajectories(buffer)

gail_trainer = GAILNoShuffle(
    demonstrations=transitions,
    demo_batch_size=1024,
    gen_replay_buffer_capacity=512,
    n_disc_updates_per_round=8,
    venv=env,
    gen_algo=learner,
    reward_net=reward_net,
    custom_logger=custom_logger,
    allow_variable_horizon=True
)

current_epoch = last_index_imitation + 1

for i in range(NUMBER_TRAIN):
    gail_trainer.train(20000)
    gail_trainer.gen_algo.save(f"./gail/steps/gail_policy{current_epoch}.zip")

    current_epoch += 1
    
gc.collect()
gail_trainer._demonstrations = None
gail_trainer._demonstrations_tensor = None
del gail_trainer

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


last_index: 2
Last index gail: -1
Using cuda device
Wrapping the env in a VecTransposeImage.


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)


obs shape: (6439, 96, 96, 1)
obs shape: (6328, 96, 96, 1)
obs shape: (6216, 96, 96, 1)
obs shape: (5931, 96, 96, 1)
obs shape: (5984, 96, 96, 1)
obs shape: (5921, 96, 96, 1)
obs shape: (5349, 96, 96, 1)
obs shape: (5794, 96, 96, 1)
obs shape: (6500, 96, 96, 1)
obs shape: (6365, 96, 96, 1)
obs shape: (5339, 96, 96, 1)
obs shape: (5293, 96, 96, 1)
obs shape: (6280, 96, 96, 1)
obs shape: (6306, 96, 96, 1)
obs shape: (5419, 96, 96, 1)
obs shape: (5598, 96, 96, 1)
obs shape: (6311, 96, 96, 1)
obs shape: (6929, 96, 96, 1)
obs shape: (6426, 96, 96, 1)
obs shape: (7318, 96, 96, 1)
obs shape: (6591, 96, 96, 1)
Running with `allow_variable_horizon` set to True. Some algorithms are biased towards shorter or longer episodes, which may significantly confound results. Additionally, even unbiased algorithms can exploit the information leak from the termination condition, producing spuriously high performance. See https://imitation.readthedocs.io/en/latest/getting-started/variable-horizon.html for mor

round:   0%|                                                                                     | 0/9 [00:00<?, ?it/s]

...

----------------------------------------------------
| raw/                               |             |
|    gen/rollout/ep_len_mean         | 5.23e+03    |
|    gen/rollout/ep_rew_mean         | 47.3        |
|    gen/rollout/ep_rew_wrapped_mean | 2.2e+03     |
|    gen/time/fps                    | 103         |
|    gen/time/iterations             | 1           |
|    gen/time/time_elapsed           | 19          |
|    gen/time/total_timesteps        | 393216      |
|    gen/train/approx_kl             | 0.015953578 |
|    gen/train/clip_fraction         | 0.157       |
|    gen/train/clip_range            | 0.2         |
|    gen/train/entropy_loss          | -1.22       |
|    gen/train/explained_variance    | 0.0747      |
|    gen/train/learning_rate         | 1e-05       |
|    gen/train/loss                  | 0.264       |
|    gen/train/n_updates             | 1910        |
|    gen/train/policy_gradient_loss  | 0.00906     |
|    gen/train/value_loss            | 1.46   

round:  33%|█████████████████████████▋                                                   | 3/9 [01:12<02:23, 23.97s/it]

----------------------------------------------------
| raw/                               |             |
|    gen/rollout/ep_len_mean         | 5.25e+03    |
|    gen/rollout/ep_rew_mean         | 47.6        |
|    gen/rollout/ep_rew_wrapped_mean | 2.2e+03     |
|    gen/time/fps                    | 104         |
|    gen/time/iterations             | 1           |
|    gen/time/time_elapsed           | 19          |
|    gen/time/total_timesteps        | 395264      |
|    gen/train/approx_kl             | 0.012613736 |
|    gen/train/clip_fraction         | 0.0689      |
|    gen/train/clip_range            | 0.2         |
|    gen/train/entropy_loss          | -1.24       |
|    gen/train/explained_variance    | 0.209       |
|    gen/train/learning_rate         | 1e-05       |
|    gen/train/loss                  | 1.07        |
|    gen/train/n_updates             | 1920        |
|    gen/train/policy_gradient_loss  | 0.0037      |
|    gen/train/value_loss            | 2.82   

round:  44%|██████████████████████████████████▏                                          | 4/9 [01:35<01:59, 23.87s/it]

----------------------------------------------------
| raw/                               |             |
|    gen/rollout/ep_len_mean         | 5.25e+03    |
|    gen/rollout/ep_rew_mean         | 47.6        |
|    gen/rollout/ep_rew_wrapped_mean | 2.17e+03    |
|    gen/time/fps                    | 102         |
|    gen/time/iterations             | 1           |
|    gen/time/time_elapsed           | 20          |
|    gen/time/total_timesteps        | 397312      |
|    gen/train/approx_kl             | 0.041966513 |
|    gen/train/clip_fraction         | 0.147       |
|    gen/train/clip_range            | 0.2         |
|    gen/train/entropy_loss          | -1.2        |
|    gen/train/explained_variance    | 0.0409      |
|    gen/train/learning_rate         | 1e-05       |
|    gen/train/loss                  | 6.02        |
|    gen/train/n_updates             | 1930        |
|    gen/train/policy_gradient_loss  | 0.00364     |
|    gen/train/value_loss            | 11.9   